# PagedCore FP32 paging accuracy

Enable **Internet** and **T4 GPU**, then **Run All**. No Input dataset, previous
notebook session, Git push or manual patch is needed. Exact source is embedded.
Set `PAGEDCORE_EVIDENCE_WORKDIR` to override the output folder; otherwise Kaggle
uses `/kaggle/working` and other environments use the current folder.

This runs CPU checks and the required T4 suite, including independent FP64
attention, owned checkpoint/prefill smoke, and paged decode versus
independent contiguous FP32 attention at the existing tolerances. Single, mixed,
fragmented and held-out layouts are required. The decoder still uses FP16 weights
and cache. No HF model forward or HF fidelity report is required. Optional HF
comparisons are deselected by default. No cuBLAS kernel profiling is needed.

Download the final ZIP even when tests fail. `accuracy-summary.json` says passed
only when CPU/GPU tests and all versioned reports are complete and valid. Setup
logs, source hashes, target identity, raw logit tensors and failures are retained.
This collector covers paged accuracy; end-to-end generation, maximum-active
workspace, CUDA reuse and service gates remain separate. Read NUMERICAL_POLICY.md
in the source snapshot for the November 11 independent validation contract. Model quality, full-distribution
KL, and expanded component coverage remain planned and are not certified here.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

PAYLOAD = "..."
SNAPSHOT_SHA256 = "0847965022829c254b20f2a7730f60442ca3ae3bd4fae8a93e553a2bb0ff7e0e"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-accuracy-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )
    run("lint", [str(python), "-m", "ruff", "check", "."], required=True)
    run("format", [str(python), "-m", "ruff", "format", "--check", "."], required=True)
    run("types", [str(python), "-m", "mypy", "--strict", "src/pagedcore"], required=True)
    run(
        "cpu-tests",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "not gpu and not benchmark",
            "-ra",
            "--junitxml=" + str(evidence / "cpu-tests.xml"),
        ],
    )
    environment["PAGEDCORE_ACCURACY_DIR"] = str(evidence / "raw-logits")
    run(
        "gpu-regressions",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "gpu",
            "-v",
            "-ra",
            "-k",
            "not legacy and not paired_at_17 and not test_pinned_candidate_paged_decode_on_t4",
            "--junitxml=" + str(evidence / "gpu-tests.xml"),
        ],
    )
    run("accuracy-export", [str(python), "-m", "scripts.export_paged_accuracy", str(evidence)])
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "notebook-failure.txt").write_text(traceback.format_exc())
    print("Run incomplete; retained the failure and partial evidence.", flush=True)
finally:
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    (evidence / "notebook-run.json").write_text(
        json.dumps(
            {
                "snapshot_sha256": SNAPSHOT_SHA256,
                "base_revision": identity["base_revision"],
                "scope": "Paged-decoder accuracy validation, not full release acceptance",
                "environment_overrides": {
                    key: environment[key]
                    for key in (
                        "CUDA_VISIBLE_DEVICES",
                        "CUBLAS_WORKSPACE_CONFIG",
                        "OMP_NUM_THREADS",
                        "MKL_NUM_THREADS",
                    )
                },
            },
            indent=2,
        )
    )
    files = {
        path.relative_to(evidence).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in evidence.rglob("*")
        if path.is_file()
    }
    (evidence / "checksums.json").write_text(json.dumps(files, indent=2))
    archive_path = working / (run_root.name + ".zip")
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("source-snapshot.zip", payload)
        for path in evidence.rglob("*"):
            if path.is_file():
                archive.write(path, "evidence/" + path.relative_to(evidence).as_posix())
    print("DOWNLOAD", archive_path, "bytes", archive_path.stat().st_size, flush=True)
    display(FileLink(str(archive_path)))